# Bài 2: Tối Ưu Hóa Hiệu Năng với Parameters trong LightGBM

## Mục tiêu bài học

Sau bài học này, bạn sẽ có thể:
- Hiểu rõ ý nghĩa và tác động của các tham số quan trọng nhất trong LightGBM.
- Phân biệt được các nhóm tham số: điều khiển cây, điều khiển tốc độ học, và điều khiển dữ liệu.
- Biết cách tinh chỉnh các tham số để cân bằng giữa độ chính xác (accuracy) và nguy cơ quá khớp (overfitting).
- Áp dụng kiến thức để cải thiện hiệu suất của mô hình LightGBM trên một bộ dữ liệu thực tế.

## Tại sao việc tinh chỉnh tham số lại quan trọng?

Ở bài học trước, chúng ta đã xây dựng một mô hình LightGBM với các tham số mặc định. Mặc dù cho kết quả không tệ, nhưng để khai thác tối đa sức mạnh của LightGBM, việc hiểu và tinh chỉnh các tham số là tối quan trọng. Giống như việc điều chỉnh một chiếc xe đua, các tham số phù hợp có thể giúp mô hình của bạn chạy nhanh hơn, chính xác hơn và ổn định hơn trên nhiều loại "địa hình" (dữ liệu) khác nhau.

Việc tinh chỉnh không đúng cách có thể dẫn đến hai vấn đề chính:
1.  **Underfitting (Dưới khớp):** Mô hình quá đơn giản, không học được các quy luật phức tạp trong dữ liệu, dẫn đến hiệu suất kém trên cả tập huấn luyện và tập kiểm tra.
2.  **Overfitting (Quá khớp):** Mô hình học thuộc lòng dữ liệu huấn luyện, bao gồm cả nhiễu. Nó hoạt động cực tốt trên tập huấn luyện nhưng lại cho kết quả rất tệ trên dữ liệu mới (tập kiểm tra).

## Các Nhóm Tham Số Chính trong LightGBM

Chúng ta có thể chia các tham số của LightGBM thành ba nhóm chính. Hiểu rõ từng nhóm sẽ giúp bạn có một chiến lược tinh chỉnh hiệu quả.

### 1. Tham số điều khiển cấu trúc cây (Tree Structure Parameters)

Đây là nhóm tham số quan trọng nhất, ảnh hưởng trực tiếp đến độ phức tạp của các cây quyết định được xây dựng.

- **`num_leaves`** (mặc định: 31)
  - **Ý nghĩa:** Số lượng lá tối đa trong một cây. Đây là tham số chính để kiểm soát độ phức tạp của mô hình.
  - **Tác động:** Tăng `num_leaves` sẽ làm tăng độ phức tạp của mô hình, giúp nó học được các quy luật phức tạp hơn, nhưng cũng làm tăng nguy cơ overfitting.
  - **Lưu ý:** Đối với LightGBM, bạn nên tập trung vào `num_leaves` thay vì `max_depth`. Mối quan hệ lý thuyết là `num_leaves` <= 2^(`max_depth`), nhưng trong thực tế, để một cây có `k` lá, nó cần độ sâu ít nhất là `log2(k)`.

- **`max_depth`** (mặc định: -1, không giới hạn)
  - **Ý nghĩa:** Độ sâu tối đa của một cây.
  - **Tác động:** Giới hạn `max_depth` giúp ngăn chặn overfitting bằng cách không cho cây phát triển quá sâu. Tuy nhiên, `num_leaves` thường là cách kiểm soát hiệu quả hơn.

- **`min_child_samples`** (mặc định: 20)
  - **Ý nghĩa:** Số lượng mẫu dữ liệu tối thiểu cần có trong một nhánh lá. Nếu một phép chia tạo ra một nhánh lá có ít hơn `min_child_samples` mẫu, phép chia đó sẽ không được thực hiện.
  - **Tác động:** Tăng giá trị này sẽ làm cho mô hình trở nên "bảo thủ" hơn, tránh tạo ra các nhánh lá chỉ học từ một vài mẫu dữ liệu, từ đó giúp chống overfitting.

- **`min_split_gain`** (mặc định: 0.0)
  - **Ý nghĩa:** Mức giảm loss (gain) tối thiểu cần có để thực hiện một phép chia. Bất kỳ phép chia nào có gain nhỏ hơn giá trị này sẽ bị bỏ qua.
  - **Tác động:** Tương tự `min_child_samples`, đây là một tham số dùng để điều chuẩn (regularization).

### 2. Tham số điều khiển quá trình học (Learning Control Parameters)

Nhóm này kiểm soát cách mô hình được cập nhật qua mỗi vòng lặp boosting.

- **`learning_rate`** (mặc định: 0.1)
  - **Ý nghĩa:** Tốc độ học. Ở mỗi bước boosting, dự đoán được cập nhật bằng cách cộng thêm `learning_rate` * (dự đoán của cây mới).
  - **Tác động:** `learning_rate` nhỏ đòi hỏi nhiều cây hơn (`n_estimators`) để đạt được cùng mức độ chính xác, nhưng thường dẫn đến một mô hình tổng quát hóa tốt hơn. Một `learning_rate` quá lớn có thể khiến mô hình "vượt" qua điểm tối ưu.
  - **Chiến lược phổ biến:** Chọn một `learning_rate` nhỏ (ví dụ: 0.01 - 0.05) và tìm số lượng cây (`n_estimators`) tối ưu bằng cách sử dụng early stopping.

- **`n_estimators`** (mặc định: 100)
  - **Ý nghĩa:** Số lượng cây (boosting rounds) sẽ được xây dựng.
  - **Tác động:** Tăng `n_estimators` sẽ làm tăng độ phức tạp của mô hình. Quá nhiều cây sẽ dẫn đến overfitting.

- **`boosting_type`** (mặc định: 'gbdt')
  - **Ý nghĩa:** Thuật toán boosting sẽ được sử dụng.
  - **Các lựa chọn:**
    - `'gbdt'`: Gradient Boosting Decision Tree truyền thống.
    - `'dart'`: Sử dụng Dropout để chống overfitting. Chậm hơn nhưng có thể chính xác hơn.
    - `'goss'`: Sử dụng Gradient-based One-Side Sampling. Nhanh hơn nhưng có thể kém chính xác hơn một chút.
    - `'rf'`: Random Forest.

### 3. Tham số điều khiển dữ liệu và I/O (Data and I/O Parameters)

Nhóm này liên quan đến cách LightGBM xử lý dữ liệu đầu vào.

- **`feature_fraction`** (mặc định: 1.0)
  - **Ý nghĩa:** Tỷ lệ các đặc trưng (features) sẽ được chọn ngẫu nhiên ở mỗi cây.
  - **Tác động:** Nếu bạn có nhiều features, đặt giá trị này nhỏ hơn 1.0 (ví dụ: 0.8) có thể giúp tăng tốc độ huấn luyện và chống overfitting. Nó tương tự như `colsample_bytree` trong XGBoost.

- **`bagging_fraction`** (mặc định: 1.0) và **`bagging_freq`** (mặc định: 0)
  - **Ý nghĩa:** `bagging_fraction` là tỷ lệ mẫu dữ liệu được lấy (không thay thế) cho mỗi vòng lặp boosting. `bagging_freq` xác định tần suất thực hiện bagging (ví dụ: `bagging_freq=5` nghĩa là cứ 5 vòng lặp thì thực hiện bagging một lần).
  - **Tác động:** Kỹ thuật này, còn gọi là subsampling, giúp tăng tốc độ và chống overfitting. Cần đặt cả hai tham số để kích hoạt.

- **`is_unbalance`** (mặc định: False) hoặc **`scale_pos_weight`**
  - **Ý nghĩa:** Dùng cho các bài toán có dữ liệu mất cân bằng (imbalanced data).
  - **Tác động:** Đặt `is_unbalance=True` sẽ tự động điều chỉnh trọng số của các lớp dựa trên tần suất của chúng. Hoặc bạn có thể tự đặt `scale_pos_weight` bằng `số lượng mẫu lớp âm / số lượng mẫu lớp dương`.

## Ví dụ thực tế: Tinh chỉnh tham số cho bài toán Churn

Chúng ta sẽ quay lại bài toán dự đoán khách hàng rời bỏ và thử nghiệm một vài bộ tham số khác nhau để xem hiệu quả.

In [ ]:
# Import các thư viện cần thiết
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, accuracy_score

# Tải và chuẩn bị dữ liệu (giống bài 1)
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop(columns=['customerID'], inplace=True)
for column in df.select_dtypes(include=['object']).columns:
    df[column] = LabelEncoder().fit_transform(df[column])

# Phân chia dữ liệu
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Thí nghiệm 1: Mô hình mặc định (Baseline)

In [ ]:
clf_default = lgb.LGBMClassifier(random_state=42)
clf_default.fit(X_train, y_train)
y_pred_default = clf_default.predict(X_test)
y_pred_proba_default = clf_default.predict_proba(X_test)[:, 1]

print("--- Mô hình Mặc định ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_default):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba_default):.4f}")

### Thí nghiệm 2: Mô hình được tinh chỉnh để chống Overfitting

Chúng ta sẽ thử một bộ tham số "bảo thủ" hơn:
- `num_leaves` nhỏ hơn.
- `max_depth` được giới hạn.
- `learning_rate` nhỏ hơn và tăng `n_estimators`.
- Sử dụng `feature_fraction` và `bagging_fraction`.

In [ ]:
params_tuned = {
    'objective': 'binary', # Bài toán phân loại nhị phân
    'metric': 'auc',       # Metric để đánh giá, phù hợp với objective
    'boosting_type': 'gbdt',
    'n_estimators': 1000,    # Tăng số cây vì learning_rate nhỏ
    'learning_rate': 0.01,   # Tốc độ học nhỏ
    'num_leaves': 20,        # Giảm so với mặc định (31)
    'max_depth': 5,          # Giới hạn độ sâu
    'seed': 42,
    'n_jobs': -1,            # Sử dụng tất cả các CPU cores
    'verbose': -1,           # Tắt bớt output khi huấn luyện
    'colsample_bytree': 0.7, # Tương đương feature_fraction
    'subsample': 0.7,        # Tương đương bagging_fraction
    'reg_alpha': 0.1,        # L1 regularization
    'reg_lambda': 0.1        # L2 regularization
}

clf_tuned = lgb.LGBMClassifier(**params_tuned)

# Sử dụng Early Stopping để tìm n_estimators tối ưu
clf_tuned.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              eval_metric='auc',
              callbacks=[lgb.early_stopping(100, verbose=False)]) # Dừng nếu AUC không cải thiện sau 100 vòng

y_pred_tuned = clf_tuned.predict(X_test)
y_pred_proba_tuned = clf_tuned.predict_proba(X_test)[:, 1]

print("\n--- Mô hình Đã Tinh Chỉnh ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba_tuned):.4f}")
print(f"Số cây tốt nhất (best_iteration): {clf_tuned.best_iteration_}")

**Phân tích kết quả:**

Bạn có thể thấy rằng mô hình đã được tinh chỉnh, mặc dù có thể có accuracy tương đương hoặc thấp hơn một chút, nhưng thường sẽ có điểm ROC AUC cao hơn. ROC AUC là một metric tốt hơn để đánh giá khả năng phân loại tổng thể của mô hình, đặc biệt là trên các dữ liệu mất cân bằng.

Quan trọng hơn, việc sử dụng `early_stopping` đã giúp chúng ta tìm ra số lượng cây tối ưu (`best_iteration_`) thay vì dùng một con số cố định (1000), giúp mô hình vừa đủ phức tạp mà không bị overfitting.

## Tóm tắt

- Tinh chỉnh tham số là bước quan trọng để tối ưu hóa mô hình LightGBM.
- Các tham số được chia thành 3 nhóm chính: điều khiển cây, điều khiển học, và điều khiển dữ liệu.
- **Để chống overfitting:** giảm `num_leaves`, `max_depth`; tăng `min_child_samples`; sử dụng `feature_fraction`, `bagging_fraction`; và sử dụng `learning_rate` nhỏ kết hợp với `early_stopping`.
- **Để tăng accuracy:** tăng `num_leaves`, `max_depth`; giảm `min_child_samples`; nhưng phải cẩn thận với overfitting.

## Bài học tiếp theo

Trong bài học tới, chúng ta sẽ khám phá các kỹ thuật xử lý dữ liệu hiệu quả dành riêng cho LightGBM, bao gồm cách xử lý các biến categorical một cách tự nhiên và các phương pháp khác để chuẩn bị dữ liệu nhằm đạt hiệu suất cao nhất.